### 📊 Experiments Results

🔹 1. Imports

In [339]:
import json
import pandas as pd
from pathlib import Path

🔹 2. Função de interpretação

In [340]:
def interpret(value, metric):
    if metric == "accuracy":
        if value > 0.8: return "🔥 Muito bom"
        elif value > 0.6: return "👍 Ok"
        else: return "⚠️ Baixo"
    
    if metric == "proxy_score":
        if value > 0.75: return "🔥 Forte"
        elif value > 0.5: return "👍 Médio"
        else: return "⚠️ Fraco"

🔹 3. Load + processamento

In [341]:
def load_and_process(path):
    data = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                continue

    rows = []

    for item in data:
        if not isinstance(item, dict):
            print("Item estranho:", item)
            continue

        pred = item.get("prediction", "").upper()
        gold = item.get("gold_label", "").upper()

        # 🔥 CORREÇÃO IMPORTANTE AQUI
        steps = item.get("pipeline", {}).get("steps", [])

        evidence_counts = [len(s.get("evidence", [])) for s in steps]

        avg_evidence = sum(evidence_counts) / len(evidence_counts) if evidence_counts else 0

        rows.append({
            "claim_id": item.get("claim_id"),
            "prediction": pred,
            "label": gold,
            "correct": pred == gold,
            "num_steps": len(steps),
            "avg_evidence_per_step": avg_evidence,
        })

    df = pd.DataFrame(rows)

    accuracy = df["correct"].mean()
    evidence_score = df["avg_evidence_per_step"].mean()

    proxy_score = 0.7 * accuracy + 0.3 * min(evidence_score / 5.0, 1.0)

    metrics = {
        "accuracy": accuracy,
        "proxy_score": proxy_score
    }

    return df, metrics

🔹 4. Paths

In [342]:
paths = {
    "baseline": [
        "../outputs/averitec_original_pipeline/2026-03-30_01-25-13/averitec_dev.jsonl"
    ],
    "iterative": [
        "../outputs/averitec_iterative_pipeline/2026-03-30_01-27-11/averitec_dev.jsonl"
    ]
}

🔹 5. Carregamento geral

In [343]:
dfs = {}
results = {}

for pipeline, file_list in paths.items():
    dfs[pipeline] = []
    results[pipeline] = []

    for path in file_list:
        df, metrics = load_and_process(path)

        run_name = Path(path).parent.name
        df["run"] = run_name

        dfs[pipeline].append(df)
        results[pipeline].append(metrics)

🔹 6. Concatenar tudo

In [344]:
dfs_concat = {
    k: pd.concat(v, ignore_index=True)
    for k, v in dfs.items()
}

🔹 7. Print por pipeline

In [ ]:
from pathlib import Path
from sklearn.metrics import classification_report

import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

def print_pipeline_summary(name, df):
    run_name = Path(path).parent.name
    dataset_name = Path(path).name

    df["run"] = run_name
    df["dataset"] = dataset_name

    for run, df_run in df.groupby("run"):

        accuracy = df_run["correct"].mean()
        evidence_score = df_run["avg_evidence_per_step"].mean()
        

        df_errors = df_run[~df_run["correct"]]

        print("=" * 50)
        print(f"📌 PIPELINE: {name.upper()} - {run}")
        #print(f"📂 ARQUIVO: {run}")
        print(f"📂 DATASET: {df_run['dataset'].iloc[0]}")
        print("\n")
        print('🎯 Accuracy:', round(accuracy, 2), '->', interpret(accuracy, 'accuracy'))
        print("\n")

        print("📊 Total samples:", len(df_run))
        print("✅ Correct predictions:", df_run["correct"].sum())
        print("❌ Errors:", len(df_errors))
        print("=" * 50)
        #print(classification_report(df["label"], df["prediction"]))
        #print("📊 Per-class accuracy:")
        #print(df.groupby("label")["correct"].mean())

for name in dfs_concat:
    print_pipeline_summary(name, dfs_concat[name])

📌 PIPELINE: BASELINE - 2026-03-30_01-27-11
📂 DATASET: averitec_dev.jsonl


🎯 Accuracy: 0.58 -> ⚠️ Baixo


📊 Total samples: 500
✅ Correct predictions: 288
❌ Errors: 212
                                    precision    recall  f1-score   support

CONFLICTING EVIDENCE/CHERRYPICKING       0.00      0.00      0.00        38
               NOT ENOUGH EVIDENCE       0.09      0.26      0.13        35
                           REFUTED       0.72      0.78      0.75       305
                         SUPPORTED       0.60      0.33      0.42       122

                          accuracy                           0.58       500
                         macro avg       0.35      0.34      0.33       500
                      weighted avg       0.59      0.58      0.57       500

📌 PIPELINE: ITERATIVE - 2026-03-30_01-27-11
📂 DATASET: averitec_dev.jsonl


🎯 Accuracy: 0.64 -> 👍 Ok


📊 Total samples: 500
✅ Correct predictions: 318
❌ Errors: 182
                                    precision    recall 

In [346]:

paths = {
    "baseline": [
        "../outputs/averitec_original_pipeline/2026-03-30_01-25-13/averitec_train.jsonl"
    ],
    "iterative": [
        "../outputs/averitec_iterative_pipeline/2026-03-30_01-27-11/averitec_train.jsonl"
    ]
}

dfs = {}
results = {}

for pipeline, file_list in paths.items():
    dfs[pipeline] = []
    results[pipeline] = []

    for path in file_list:
        df, metrics = load_and_process(path)

        run_name = Path(path).parent.name
        df["run"] = run_name

        dfs[pipeline].append(df)
        results[pipeline].append(metrics)

dfs_concat = {
    k: pd.concat(v, ignore_index=True)
    for k, v in dfs.items()
}

for name in dfs_concat:
    print_pipeline_summary(name, dfs_concat[name])

📌 PIPELINE: BASELINE - 2026-03-30_01-27-11
📂 DATASET: averitec_train.jsonl


🎯 Accuracy: 0.53 -> ⚠️ Baixo


📊 Total samples: 1224
✅ Correct predictions: 654
❌ Errors: 570
                                    precision    recall  f1-score   support

CONFLICTING EVIDENCE/CHERRYPICKING       0.00      0.00      0.00        65
               NOT ENOUGH EVIDENCE       0.09      0.30      0.14        93
                           REFUTED       0.73      0.70      0.71       792
                         SUPPORTED       0.50      0.27      0.35       274

                          accuracy                           0.53      1224
                         macro avg       0.33      0.32      0.30      1224
                      weighted avg       0.59      0.53      0.55      1224

📌 PIPELINE: ITERATIVE - 2026-03-30_01-27-11
📂 DATASET: averitec_train.jsonl


🎯 Accuracy: 0.56 -> ⚠️ Baixo


📊 Total samples: 333
✅ Correct predictions: 188
❌ Errors: 145
                                    precision  